In [1]:
pip install pandas numpy requests ccxt scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import time
import math
import requests
import numpy as np
import pandas as pd
import ccxt

GAMMA_URL = "https://gamma-api.polymarket.com"
CLOB_URL = "https://clob.polymarket.com"
DATA_URL = "https://data-api.polymarket.com"

MARKET_SLUG = "bitcoin-above-80000-on-march-31"   # example placeholder
BTC_SYMBOL = "BTC/USDT"
BTC_TIMEFRAME = "5m"
START = "2026-01-01T00:00:00Z"
END = "2026-03-21T00:00:00Z"

# -----------------------------
# Helpers
# -----------------------------
def to_unix_s(ts: str) -> int:
    return int(pd.Timestamp(ts, tz="UTC").timestamp())

def safe_get(url, params=None, timeout=20):
    r = requests.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    return r.json()

# -----------------------------
# 1) Polymarket market discovery
# -----------------------------
def fetch_market_by_slug(slug: str):
    # Gamma docs show public /markets and market discovery via slug-oriented lookups/workflows.
    # We use /markets and filter client-side to stay robust.
    offset = 0
    limit = 100
    while True:
        data = safe_get(
            f"{GAMMA_URL}/markets",
            params={"active": "true", "closed": "false", "limit": limit, "offset": offset},
        )
        if not data:
            break

        for m in data:
            if m.get("slug") == slug:
                return m

        if len(data) < limit:
            break
        offset += limit

    raise ValueError(f"Market slug not found: {slug}")

def extract_token_ids(market_json):
    """
    Polymarket markets are binary and map to token IDs for Yes/No outcomes.
    The exact key shape can vary across responses, so handle common cases.
    """
    token_ids = []

    # common patterns
    for key in ["tokens", "outcomes", "clobTokenIds", "token_ids"]:
        if key in market_json and market_json[key]:
            value = market_json[key]
            if isinstance(value, list):
                for x in value:
                    if isinstance(x, dict):
                        tid = x.get("token_id") or x.get("asset_id") or x.get("id")
                        if tid:
                            token_ids.append(tid)
                    elif isinstance(x, str):
                        token_ids.append(x)

    # de-dup preserve order
    seen = set()
    out = []
    for x in token_ids:
        if x not in seen:
            seen.add(x)
            out.append(x)

    if len(out) < 2:
        raise ValueError("Could not reliably extract Yes/No token IDs from market JSON.")
    return out[:2]

# -----------------------------
# 2) Polymarket price history
# -----------------------------
def fetch_poly_price_history(token_id: str, start_ts: int, end_ts: int, interval="1m", fidelity=1) -> pd.DataFrame:
    data = safe_get(
        f"{CLOB_URL}/prices-history",
        params={
            "market": token_id,   # docs use 'market' query param for asset/token id on this endpoint
            "startTs": start_ts,
            "endTs": end_ts,
            "interval": interval,
            "fidelity": fidelity,
        },
    )
    hist = data.get("history", [])
    if not hist:
        return pd.DataFrame(columns=["timestamp", "poly_price"])

    df = pd.DataFrame(hist)
    # docs example uses keys t, p
    df = df.rename(columns={"t": "timestamp", "p": "poly_price"})
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
    df["poly_price"] = pd.to_numeric(df["poly_price"], errors="coerce")
    return df[["timestamp", "poly_price"]].dropna().sort_values("timestamp").reset_index(drop=True)

# -----------------------------
# 3) Polymarket current orderbook
# -----------------------------
def fetch_current_orderbook(token_id: str) -> dict:
    return safe_get(f"{CLOB_URL}/book", params={"token_id": token_id})

def top_of_book_features(book_json: dict) -> dict:
    bids = book_json.get("bids", [])
    asks = book_json.get("asks", [])

    best_bid = float(bids[0]["price"]) if bids else np.nan
    best_ask = float(asks[0]["price"]) if asks else np.nan
    bid_size = float(bids[0]["size"]) if bids else np.nan
    ask_size = float(asks[0]["size"]) if asks else np.nan

    spread = best_ask - best_bid if np.isfinite(best_bid) and np.isfinite(best_ask) else np.nan
    mid = (best_bid + best_ask) / 2 if np.isfinite(best_bid) and np.isfinite(best_ask) else np.nan
    imbalance = bid_size / (bid_size + ask_size) if np.isfinite(bid_size) and np.isfinite(ask_size) and (bid_size + ask_size) > 0 else np.nan

    return {
        "best_bid": best_bid,
        "best_ask": best_ask,
        "mid": mid,
        "spread": spread,
        "bid_size": bid_size,
        "ask_size": ask_size,
        "imbalance": imbalance,
    }

# -----------------------------
# 4) Binance BTC history
# -----------------------------
def fetch_binance_ohlcv_paginated(symbol: str, timeframe: str, since_iso: str, end_iso: str) -> pd.DataFrame:
    exchange = ccxt.binance()
    since = exchange.parse8601(since_iso)
    end_ms = exchange.parse8601(end_iso)
    limit = 1000
    rows = []

    while since < end_ms:
        batch = exchange.fetch_ohlcv(symbol, timeframe=timeframe, since=since, limit=limit)
        if not batch:
            break

        rows.extend(batch)
        last_ts = batch[-1][0]
        since = last_ts + 1

        if len(batch) < limit:
            break

        time.sleep(exchange.rateLimit / 1000)

    df = pd.DataFrame(rows, columns=["timestamp", "open", "high", "low", "close", "volume"])
    if df.empty:
        return df

    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
    df = df[df["timestamp"] <= pd.Timestamp(end_iso, tz="UTC")].copy()
    return df.drop_duplicates(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

# -----------------------------
# 5) Feature engineering
# -----------------------------
def prepare_merged_df(poly_df: pd.DataFrame, btc_df: pd.DataFrame, rule="YES"):
    """
    rule="YES" means higher BTC should generally help the token price.
    If the market is a DOWN / BELOW contract, invert later.
    """
    # Resample both to 5-minute close-aligned points
    poly = poly_df.set_index("timestamp").resample("5min").last().ffill().reset_index()
    btc = btc_df.set_index("timestamp")[["close"]].resample("5min").last().ffill().reset_index()
    btc = btc.rename(columns={"close": "btc_close"})

    df = pd.merge(poly, btc, on="timestamp", how="inner").dropna().copy()

    # returns
    df["poly_ret"] = df["poly_price"].pct_change()
    df["btc_ret"] = df["btc_close"].pct_change()

    # invert BTC sign for markets that benefit from BTC falling
    if rule.upper() == "NO" or rule.upper() == "DOWN":
        df["btc_ret_effective"] = -df["btc_ret"]
    else:
        df["btc_ret_effective"] = df["btc_ret"]

    return df.dropna().reset_index(drop=True)

def fit_beta(df: pd.DataFrame, train_frac=0.6):
    n = len(df)
    split = max(30, int(n * train_frac))
    train = df.iloc[:split].copy()
    x = train["btc_ret_effective"].values
    y = train["poly_ret"].values

    # OLS beta through covariance
    x_var = np.var(x)
    beta = np.cov(x, y, ddof=0)[0, 1] / x_var if x_var > 0 else 0.0
    return beta, split

def add_signal_columns(df: pd.DataFrame, beta: float):
    df = df.copy()
    df["fair_poly_ret"] = beta * df["btc_ret_effective"]
    df["residual"] = df["poly_ret"] - df["fair_poly_ret"]

    # rolling z-score of residual
    lookback = 48  # 4 hours on 5m bars
    rolling_mean = df["residual"].rolling(lookback).mean()
    rolling_std = df["residual"].rolling(lookback).std()

    df["resid_z"] = (df["residual"] - rolling_mean) / (rolling_std + 1e-12)

    # extra features
    df["poly_mom_3"] = df["poly_price"].pct_change(3)
    df["btc_mom_3"] = df["btc_close"].pct_change(3)
    return df

# -----------------------------
# 6) Backtest
# -----------------------------
def backtest_mean_reversion(
    df: pd.DataFrame,
    start_index: int,
    entry_z=-1.8,
    exit_z=-0.2,
    stop_loss=-0.08,
    take_profit=0.12,
    max_holding_bars=12,
):
    """
    Long-only: buy Polymarket when token underreacts/overreacts downward vs BTC fair move.
    Use simple close-to-close backtest on price history.
    """
    trades = []
    in_pos = False
    entry_price = None
    entry_time = None
    entry_idx = None

    for i in range(start_index, len(df)):
        row = df.iloc[i]
        price = row["poly_price"]
        z = row["resid_z"]

        if not np.isfinite(z):
            continue

        if not in_pos:
            if z <= entry_z:
                in_pos = True
                entry_price = price
                entry_time = row["timestamp"]
                entry_idx = i
        else:
            pnl = (price / entry_price) - 1
            holding = i - entry_idx

            should_exit = (
                z >= exit_z or
                pnl <= stop_loss or
                pnl >= take_profit or
                holding >= max_holding_bars
            )

            if should_exit:
                trades.append({
                    "entry_time": entry_time,
                    "exit_time": row["timestamp"],
                    "entry_price": entry_price,
                    "exit_price": price,
                    "bars_held": holding,
                    "return": pnl,
                    "exit_z": z,
                })
                in_pos = False
                entry_price = None
                entry_time = None
                entry_idx = None

    trades_df = pd.DataFrame(trades)
    if trades_df.empty:
        return trades_df, {}

    eq = (1 + trades_df["return"]).cumprod()
    stats = {
        "n_trades": len(trades_df),
        "win_rate": float((trades_df["return"] > 0).mean()),
        "avg_return": float(trades_df["return"].mean()),
        "median_return": float(trades_df["return"].median()),
        "cum_return_multiple": float(eq.iloc[-1]),
    }
    return trades_df, stats

# -----------------------------
# 7) Run
# -----------------------------
if __name__ == "__main__":
    start_ts = to_unix_s(START)
    end_ts = to_unix_s(END)

    market = fetch_market_by_slug(MARKET_SLUG)
    token_ids = extract_token_ids(market)

    # You must decide which token corresponds to the direction you want.
    # Usually token_ids[0]/[1] must be checked against outcome labels in the market payload.
    target_token = token_ids[0]

    poly_hist = fetch_poly_price_history(target_token, start_ts, end_ts, interval="1m", fidelity=1)
    btc_hist = fetch_binance_ohlcv_paginated(BTC_SYMBOL, BTC_TIMEFRAME, START, END)

    if poly_hist.empty:
        raise RuntimeError("No Polymarket historical price data returned.")
    if btc_hist.empty:
        raise RuntimeError("No BTC history returned.")

    df = prepare_merged_df(poly_hist, btc_hist, rule="YES")
    beta, split_idx = fit_beta(df)
    df = add_signal_columns(df, beta)

    trades, stats = backtest_mean_reversion(
        df,
        start_index=split_idx,
        entry_z=-1.8,
        exit_z=-0.2,
        stop_loss=-0.08,
        take_profit=0.12,
        max_holding_bars=12,
    )

    print("Estimated beta:", beta)
    print("Backtest stats:", stats)

    if not trades.empty:
        print(trades.head())

    # Optional: current execution filter
    current_book = fetch_current_orderbook(target_token)
    tob = top_of_book_features(current_book)
    print("Top of book:", tob)